<a href="https://colab.research.google.com/github/FC-Andrade/Analises-complementares/blob/main/PCA_vs_NMA_Comparison_%E2%80%93_Integrin_Essential_Dynamics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# PCA vs NMA Comparison – Integrin Essential Dynamics
# =========================================================
# Dependencies: numpy, pandas, matplotlib, seaborn, scipy
# Optional: mdtraj or MDAnalysis (para leitura dos dados MD)
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import svd

sns.set(style="whitegrid", context="talk")

# =====================================
# 1️⃣ ENTRADA: Matrizes de autovetores
# =====================================
# Cada coluna é um vetor normalizado representando um modo
# -> pcs: obtidos da análise PCA das trajetórias MD
# -> modes: obtidos da NMA

# Exemplo com dados fictícios (substitua pelos seus):
np.random.seed(1)
pcs = np.random.randn(1000, 7)   # 7 principais componentes
modes = np.random.randn(1000, 5) # 5 modos normais

# Normalizar os vetores coluna a coluna
pcs /= np.linalg.norm(pcs, axis=0)
modes /= np.linalg.norm(modes, axis=0)

# =====================================
# 2️⃣ VARIÂNCIA EXPLICADA (painéis A e B)
# =====================================
var_pca = np.linspace(0.4, 0.01, 20)
var_pca /= var_pca.sum()
cum_pca = np.cumsum(var_pca)

var_nma = np.linspace(0.25, 0.005, 20)
var_nma /= var_nma.sum()
cum_nma = np.cumsum(var_nma)

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

axs[0].bar(range(1, 21), var_pca, color="royalblue")
axs[0].plot(range(1, 21), cum_pca, "o-", color="royalblue")
axs[0].set_title("A  Fraction of variance (PCA)")
axs[0].set_xlabel("PCA index")
axs[0].set_ylabel("Fraction of variance")
axs[0].axhline(0.8, ls="--", color="k")
axs[0].axvline(5, ls="--", color="k")

axs[1].bar(range(1, 21), var_nma, color="royalblue")
axs[1].plot(range(1, 21), cum_nma, "o-", color="royalblue")
axs[1].set_title("B  Fraction of variance (Normal Modes)")
axs[1].set_xlabel("Mode index")
axs[1].axhline(0.8, ls="--", color="k")
axs[1].axvline(5, ls="--", color="k")

plt.tight_layout()
plt.show()

# =====================================
# 3️⃣ MATRIZ DE OVERLAP (painel C)
# =====================================
overlap = np.abs(np.dot(pcs.T, modes))
overlap_df = pd.DataFrame(
    overlap,
    index=[f"PC{i+1}" for i in range(overlap.shape[0])],
    columns=[f"Mode {i+1}" for i in range(overlap.shape[1])]
)

plt.figure(figsize=(6, 4))
sns.heatmap(overlap_df, annot=True, fmt=".2f", cmap="Blues", cbar=False)
plt.title("C  Overlap between PCA and NMA modes")
plt.show()

# =====================================
# 4️⃣ RMSIP (painel D)
# =====================================
def rmsip(A, B, n):
    """Root Mean Square Inner Product entre subespaços A e B"""
    U, s, Vt = svd(np.dot(A[:, :n].T, B[:, :n]))
    return np.sqrt(np.mean(s**2))

max_modes = 5
rmsip_matrix = np.zeros((pcs.shape[1], max_modes))

for i in range(pcs.shape[1]):
    for j in range(max_modes):
        rmsip_matrix[i, j] = rmsip(pcs, modes, min(i+1, j+1))

rmsip_df = pd.DataFrame(
    rmsip_matrix,
    index=[f"PC{i+1}" for i in range(pcs.shape[1])],
    columns=[f"Mode 1-{j+1}" for j in range(max_modes)]
)

plt.figure(figsize=(6, 4))
sns.heatmap(rmsip_df, annot=True, fmt=".2f", cmap="Blues", cbar=False)
plt.title("D  RMSIP between PCA and NMA subspaces")
plt.show()
